In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field

In [2]:
load_dotenv(override=True)

True

In [3]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if  openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ
OpenRouter API Key exists and begins sk-or-
Groq API Key exists and begins gsk_


In [4]:
instructions = """
You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

In [5]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [6]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

In [7]:
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
oss_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

In [8]:
sales_agent1 = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
sales_agent2 = Agent(name="GPT-OSS Sales Agent",instructions=instructions, model=oss_model)

In [9]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)

In [12]:
from messenger import send_email

def send_message(subject, text_body, html_body):
    send_email(subject, text_body, html_body)

In [13]:
send_message("Yet another test", "Hooray!", "<html><body><h1>Hooray!</h1></body></html>")

In [25]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects

    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

In [26]:
tools = [tool1, tool2, send_email_tool]

In [27]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_agent tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.

3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-5.4-mini")

In [28]:
with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)

Sent the best email.


# Structured Outputs

In [30]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [31]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [32]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Faith
"""

In [33]:
checker = Agent(name="Checker", instructions="You review potential sales emails", model="gpt-5.4-mini", output_type=EmailReview)
result = await Runner.run(checker, email)

In [34]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=3, contains_placeholders=True)

In [35]:
review.is_professional

False

# Guardrails

In [36]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review},tripwire_triggered=is_problem)

In [37]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model=gemini_model, output_guardrails=[email_guardrail])

In [38]:
result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
result.final_output

OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire

In [39]:
# Simpler approach

simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model=gemini_model)
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)

Howdy partner. If you’re lookin’ to stop wranglin’ with messy data and start rustlin’ up some real insights, you’ve come to the right corral. 

Here’s the email I’d send to get ’em in the saddle:

***

**Subject: Lassoing your data chaos, [Prospect Name]**

Howdy [Prospect Name],

Most folks I talk to in [Industry] feel like they’re tryin’ to herd cats when it comes to their data. You’ve got all the pieces of the puzzle, but they’re scattered across the plains and causin’ more headaches than a long ride in the sun.

At ComplAI, we’ve built a rig that does the heavy liftin’ for ya. We help teams like yours rope in their compliance and data headaches so you can get back to focusin’ on the trail ahead, rather than fixin’ fences all day.

Got a moment later this week to chat about how we can help you round up those operational inefficiencies? I promise to keep it shorter than a horse-fly’s temper.

Best trails,

[Your Name]
Sales Manager, ComplAI

***

How’s that sit with ya, partner? Read

In [40]:
result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good")

The email is not professional or has placeholders and will not be sent


# Exercise

In [41]:
class Email(BaseModel):
    subject_line: str = Field(description="The subject line of the email")
    body_text: str = Field(description="The main message of the email")
    signature_lines: list[str] = Field(description="A block of text that appears at the bottom of an email that includes details like the sender's name, title, and contact information")

In [42]:
email_agent = Agent(name="Email Generator", instructions="You write professional and appropriate emails. You should return the structured Email object instead of unstructured free text.", model=gemini_model, output_type=Email)

In [43]:
task = """
Pitch a new technological product in the form of an email that is convincing and professional to only one potential client.
"""

In [44]:
result = await Runner.run(email_agent, task)

In [45]:
review = result.final_output
review

Email(subject_line='Optimizing Operational Efficiency with NexusFlow AI', body_text="I hope this email finds you well. I have been following your team's impressive growth at InnovateCorp and wanted to introduce you to NexusFlow AI, our latest platform designed to automate complex data integration tasks. Our current partners have reported a 30% reduction in processing time and a significant decrease in manual error rates within the first quarter of implementation. Given your focus on scaling infrastructure this year, I believe NexusFlow could provide the leverage you need to streamline your operations. I would appreciate the opportunity to provide a brief, 15-minute demonstration of how our solution can be tailored to your specific workflows.", signature_lines=['Best regards,', 'Alex Rivera', 'Senior Account Executive', 'NexusFlow Solutions', 'alex.rivera@nexusflow.com'])

In [48]:
# converting email object to JSON string
email_json = result.final_output.model_dump_json()

print(email_json)

{"subject_line":"Optimizing Operational Efficiency with NexusFlow AI","body_text":"I hope this email finds you well. I have been following your team's impressive growth at InnovateCorp and wanted to introduce you to NexusFlow AI, our latest platform designed to automate complex data integration tasks. Our current partners have reported a 30% reduction in processing time and a significant decrease in manual error rates within the first quarter of implementation. Given your focus on scaling infrastructure this year, I believe NexusFlow could provide the leverage you need to streamline your operations. I would appreciate the opportunity to provide a brief, 15-minute demonstration of how our solution can be tailored to your specific workflows.","signature_lines":["Best regards,","Alex Rivera","Senior Account Executive","NexusFlow Solutions","alex.rivera@nexusflow.com"]}


In [49]:
# converting email object to Python dictionary
email_dict = result.final_output.model_dump()

print(email_dict)

{'subject_line': 'Optimizing Operational Efficiency with NexusFlow AI', 'body_text': "I hope this email finds you well. I have been following your team's impressive growth at InnovateCorp and wanted to introduce you to NexusFlow AI, our latest platform designed to automate complex data integration tasks. Our current partners have reported a 30% reduction in processing time and a significant decrease in manual error rates within the first quarter of implementation. Given your focus on scaling infrastructure this year, I believe NexusFlow could provide the leverage you need to streamline your operations. I would appreciate the opportunity to provide a brief, 15-minute demonstration of how our solution can be tailored to your specific workflows.", 'signature_lines': ['Best regards,', 'Alex Rivera', 'Senior Account Executive', 'NexusFlow Solutions', 'alex.rivera@nexusflow.com']}
